[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fgram-devAI/nlp-ncsr-task3/blob/main/notebooks/06_q6_pos_tagging.ipynb)

# Q6 — POS tagging with BERT (CoNLL-2003)

NCSR Athens, NLP Assignment 3, Question 6. Self-contained Colab notebook.

**Modification of `src/NER-BERT.py` (the assignment starter):**
- Tag column switched from `ner_tags` → `pos_tags` (~45 Penn-Treebank classes instead of 9 BIO).
- Entity-level metrics dropped per **footnote 6** of the assignment: POS tags carry no BIO span structure, so seqeval cannot evaluate them. `EvaluateModel`, `report_metrics`, and the per-seed JSON shape are simplified accordingly.
- Token-level metrics (sklearn micro accuracy, balanced accuracy, per-class classification report) are kept and are the only metrics reported.
- Same 3-seed loop (`SEEDS = [42, 43, 44]`), same hyperparameters as Q1.
- Per-seed metrics JSON written to `/content/results/q6/seed_<seed>.json`.

**Runtime**: `Runtime → Change runtime type → T4 GPU`. Each seed takes ~10–15 min on T4 (similar to Q1 because this is a full fine-tune, not head-only).

In [ ]:
# Silence Pylance/pyright stub-strictness for this Colab-only notebook.
# Stub issues from torch / transformers / sklearn are not runtime bugs.
# pyright: reportArgumentType=false, reportAttributeAccessIssue=false, reportPrivateImportUsage=false, reportReturnType=false

## 1. Environment check

In [ ]:
!nvidia-smi || echo 'NO GPU — switch to a GPU runtime before continuing.'

## 2. Install dependencies
Colab ships `torch`, `transformers`, `sklearn`, `tqdm`. Q6 needs `kagglehub` (dataset download). `seqeval` is **not** required for Q6 since we skip entity-level metrics per footnote 6 — install it only if you want to keep your notebook installs uniform with Q1/Q5.

In [ ]:
%pip install -q 'kagglehub<1.0'

## 3. Kaggle credentials → environment
Set the secrets in Colab (key icon in left sidebar): `KAGGLE_USERNAME`, `KAGGLE_KEY`. Then run this cell once per session.

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print('Kaggle user loaded:', os.environ['KAGGLE_USERNAME'])

## 4. Download CoNLL-2003

In [ ]:
import kagglehub
from pathlib import Path

dataset_path = Path(kagglehub.dataset_download('alaakhaled/conll003-englishversion'))

def _locate(stem: str) -> Path:
    hits = list(dataset_path.rglob(f'{stem}.txt'))
    if not hits:
        raise FileNotFoundError(
            f'Could not find {stem}.txt under {dataset_path}. '
            'The Kaggle dataset layout may have changed; inspect the directory listing.'
        )
    return hits[0]

train_file = _locate('train')
valid_file = _locate('valid')
test_file = _locate('test')
print('train:', train_file)
print('valid:', valid_file)
print('test :', test_file)

# Sanity-check the split sizes against the assignment preliminaries (PDF p.2):
# train=14,041, valid=3,250, test=3,453.
expected = {'train': 14041, 'valid': 3250, 'test': 3453}
for label, path in (('train', train_file), ('valid', valid_file), ('test', test_file)):
    sent_count = sum(1 for line in open(path) if line.startswith('-DOCSTART-') or not line.strip()) - 1
    print(f'{label}: {sent_count} sentence boundaries (expected ~{expected[label]})')

## 5. Imports + hyperparameters

In [ ]:
import json
import random
import statistics
import subprocess
import time

import numpy as np
import torch
import torch.optim as optim
import transformers
from transformers import AutoTokenizer, BertForTokenClassification
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from tqdm.auto import tqdm

# Pin a record of the runtime versions so the report can cite them.
print(f'torch        {torch.__version__}')
print(f'transformers {transformers.__version__}')
print(f'numpy        {np.__version__}')

# hyper-parameters (identical to Q1 for a fair comparison)
EPOCHS = 3
BATCH_SIZE = 8
LR = 1e-5
SEEDS = [42, 43, 44]

# device select: CUDA on Colab, MPS local, CPU fallback
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('device:', device)

# results location (paste the Drive-mount snippet from the project README BEFORE
# this cell if you want results persisted to Google Drive instead of /content)
RESULTS_DIR = Path('/content/results/q6')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 5b. (Optional) Persist results to Google Drive

Mount Drive so per-seed JSONs survive Colab runtime kills (idle timeout, hardware preemption, browser close). If you skip this cell, results land at `/content/results/q6/` (ephemeral — gone with the runtime). If you run it, all writes downstream redirect to `MyDrive/nlp-ncsr-task3-results/q6/` automatically.

In [ ]:
# Mount Google Drive (first run shows an OAuth popup — accept once per session).
from google.colab import drive
drive.mount('/content/drive')

# Redirect RESULTS_DIR through Drive. Everything downstream writes here
# without any further changes — the seed loop uses `RESULTS_DIR / f'seed_{seed}.json'`.
RESULTS_DIR = Path('/content/drive/MyDrive/nlp-ncsr-task3-results/q6')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('per-seed JSONs will now be saved to:', RESULTS_DIR)

## 6. Load CoNLL-2003 sentences
Verbatim from `NER-BERT.py` — `load_sentences` parses all four columns (token, POS, chunk, NER); Q6 just picks `pos_tags` downstream.

In [ ]:
def load_sentences(filepath):
    sentences = []
    tokens, pos_tags, chunk_tags, ner_tags = [], [], [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f.readlines():
            if line.startswith('-DOCSTART-') or line.strip() == '':
                if len(tokens) > 0:
                    sentences.append({
                        'tokens': tokens, 'pos_tags': pos_tags,
                        'chunk_tags': chunk_tags, 'ner_tags': ner_tags,
                    })
                    tokens, pos_tags, chunk_tags, ner_tags = [], [], [], []
            else:
                l = line.strip().split(' ')
                if len(l) >= 4:
                    tokens.append(l[0]); pos_tags.append(l[1])
                    chunk_tags.append(l[2]); ner_tags.append(l[3])
    if len(tokens) > 0:
        sentences.append({
            'tokens': tokens, 'pos_tags': pos_tags,
            'chunk_tags': chunk_tags, 'ner_tags': ner_tags,
        })
    return sentences

print('loading data')
train_sentences = load_sentences(train_file)
valid_sentences = load_sentences(valid_file)
test_sentences = load_sentences(test_file)
print(f'train={len(train_sentences)}, valid={len(valid_sentences)}, test={len(test_sentences)}')

## 7. Label vocabulary — Q6 reads the POS column
Expect ~45 distinct Penn-Treebank tags (NN, NNS, NNP, VBZ, VBD, JJ, RB, IN, DT, CD, TO, …) — 5× more classes than Q1's 9 BIO NER tags.

In [ ]:
all_tags = sorted({tag for s in train_sentences for tag in s['pos_tags']})
label2id = {tag: i for i, tag in enumerate(all_tags)}
id2label = {i: tag for tag, i in label2id.items()}
num_labels = len(all_tags)
print('Tagset size:', num_labels)
print('Tags:', all_tags)

## 8. Tokenizer + label alignment
`align_label` is identical to `NER-BERT.py` (works for any per-token tag column). `encode` reads `sentence['pos_tags']` instead of `sentence['ner_tags']` — that is the only Q6 change in this cell.

In [ ]:
bert_version = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(bert_version)

def align_label(tokens, labels):
    word_ids = tokens.word_ids()
    previous_word_idx = None
    label_ids = []
    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(label2id.get(labels[word_idx], -100))
        else:
            label_ids.append(-100)
        previous_word_idx = word_idx
    return label_ids

def encode(sentence):
    encodings = tokenizer(
        sentence['tokens'],
        truncation=True,
        padding='max_length',
        is_split_into_words=True,
        return_tensors='pt',
    )
    labels = align_label(encodings, sentence['pos_tags'])
    return {
        'input_ids': encodings['input_ids'].squeeze(0),
        'attention_mask': encodings['attention_mask'].squeeze(0),
        'labels': torch.tensor(labels, dtype=torch.long),
    }

print('encoding data')
train_dataset_raw = [encode(s) for s in train_sentences]
valid_dataset_raw = [encode(s) for s in valid_sentences]
test_dataset_raw = [encode(s) for s in test_sentences]

class InputDataset(torch.utils.data.Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

train_dataset = InputDataset(train_dataset_raw)
valid_dataset = InputDataset(valid_dataset_raw)
test_dataset = InputDataset(test_dataset_raw)

## 9. Evaluation helpers — token-level only (no seqeval)
Per footnote 6: POS tags have no BIO span structure, so seqeval is dropped. `EvaluateModel` now only accumulates flat token arrays for sklearn; `report_metrics` only prints the token-level block.

In [ ]:
def EvaluateModel(model, data_loader):
    model.eval()
    Y_actual_flat, Y_preds_flat = [], []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Evaluating'):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=-1)
            for idx in range(batch['labels'].size(0)):
                true_values_all = batch['labels'][idx]
                mask = (true_values_all != -100)
                Y_actual_flat.append(true_values_all[mask])
                Y_preds_flat.append(preds[idx][mask])
    Y_actual_flat = torch.cat(Y_actual_flat).detach().cpu().numpy()
    Y_preds_flat = torch.cat(Y_preds_flat).detach().cpu().numpy()
    return Y_actual_flat, Y_preds_flat


def report_metrics(Y_actual, Y_preds, split_name):
    print(f'\n=== {split_name} — Token-level metrics ===')
    print('Accuracy          : {:.3f}'.format(accuracy_score(Y_actual, Y_preds)))
    print('Balanced accuracy : {:.3f}'.format(balanced_accuracy_score(Y_actual, Y_preds)))

## 10. Seed control + git SHA helper

In [ ]:
def set_seeds(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

def _git_sha():
    try:
        return subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None

GIT_SHA = _git_sha()
PER_SEED_RESULTS = []

## 11. The Q6 change — same Q1 training pipeline, POS tags as target

No freezing (this is a full fine-tune). The only Q6-specific deltas inside the seed loop vs Q1 are:
- `EvaluateModel` returns 2 arrays instead of 4 (no per-sentence BIO lists).
- `report_metrics` takes 3 args instead of 5 (no entity print block).
- The per-seed JSON `test` dict has 2 keys instead of 4 (no `entity_micro_f1`, no `entity_macro_f1`).

`num_labels` is much larger here (~45 vs Q1's 9), so the classifier head is wider. The encoder cost still dominates wall-time.

In [ ]:
for run_index, seed in enumerate(SEEDS):
    print(f'\n{"#" * 60}\n# Q6 run {run_index + 1}/{len(SEEDS)} — seed={seed}\n{"#" * 60}')

    set_seeds(seed)

    # fresh model + head per seed
    print('initializing the model')
    model = BertForTokenClassification.from_pretrained(
        bert_version,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
    )

    model = model.to(device)
    optimizer = optim.AdamW(params=model.parameters(), lr=LR)

    train_generator = torch.Generator().manual_seed(seed)
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=train_generator,
    )
    valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=BATCH_SIZE)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE)

    print('training the model')
    train_start = time.perf_counter()
    for epoch in range(EPOCHS):
        model.train()
        print(f'epoch {epoch + 1}/{EPOCHS}')
        for batch in tqdm(train_loader, desc=f'Training epoch {epoch + 1}'):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        Y_actual, Y_preds = EvaluateModel(model, valid_loader)
        report_metrics(Y_actual, Y_preds, split_name=f'Validation (epoch {epoch + 1})')
    training_seconds = time.perf_counter() - train_start

    print(f'\napplying the model to the test set (seed={seed})')
    Y_actual, Y_preds = EvaluateModel(model, test_loader)
    report_metrics(Y_actual, Y_preds, split_name=f'Test (seed={seed})')

    label_ids_sorted = list(range(num_labels))
    target_names = [id2label[i] for i in label_ids_sorted]
    print(f'\n=== Test (seed={seed}) — Token-level classification report (sklearn) ===')
    print(classification_report(Y_actual, Y_preds, labels=label_ids_sorted,
                                target_names=target_names, zero_division=0))

    # NOTE on JSON shape: top-level `script` matches Q1's key for aggregator parity.
    # `notebook` is ALSO included so the JSON traces back to the Colab runtime.
    # Per footnote 6, the `test` dict carries only token-level metrics.
    metrics = {
        'question': 'Q6',
        'script': 'src/q6_pos_tagging.py',
        'notebook': 'notebooks/06_q6_pos_tagging.ipynb',
        'model': bert_version,
        'task': 'pos',
        'tag_column': 'pos_tags',
        'seed': seed,
        'run_index': run_index,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'training_seconds': training_seconds,
        'num_labels': num_labels,
        'test': {
            'token_micro_accuracy': float(accuracy_score(Y_actual, Y_preds)),
            'token_macro_accuracy': float(balanced_accuracy_score(Y_actual, Y_preds)),
        },
        'device': str(device),
        'git_commit': GIT_SHA,
    }
    out_path = RESULTS_DIR / f'seed_{seed}.json'
    out_path.write_text(json.dumps(metrics, indent=2) + '\n')
    print(f'\nsaved metrics: {out_path}')
    PER_SEED_RESULTS.append(metrics)

    del model, optimizer, train_loader, valid_loader, test_loader
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    if torch.backends.mps.is_available(): torch.mps.empty_cache()

## 12. Summary across seeds (token-level only)

In [ ]:
print(f'\n{"#" * 60}\n# Q6 — summary across {len(SEEDS)} seeds\n{"#" * 60}')
metric_keys = ['token_micro_accuracy', 'token_macro_accuracy']
print(f"{'metric':30s}  {'mean':>10s}  {'stdev':>10s}  values")
for k in metric_keys:
    vals = [r['test'][k] for r in PER_SEED_RESULTS]
    m = statistics.mean(vals)
    s = statistics.stdev(vals) if len(vals) > 1 else 0.0
    vals_str = ', '.join(f'{v:.4f}' for v in vals)
    print(f'{k:30s}  {m:>10.4f}  {s:>10.4f}  [{vals_str}]')
times = [r['training_seconds'] for r in PER_SEED_RESULTS]
print(f"{'training_seconds':30s}  {statistics.mean(times):>10.1f}  {statistics.stdev(times) if len(times) > 1 else 0.0:>10.1f}  [{', '.join(f'{t:.1f}' for t in times)}]")

## 14. Download results to local disk (skip if Drive mount above ran)

If you ran cell §5b, results are already in Drive — no download needed. Otherwise each `files.download()` call opens a browser-native save dialog; drop the files into `results/q6/` in your local repo and commit.

In [ ]:
from google.colab import files
for f in sorted(RESULTS_DIR.glob('seed_*.json')):
    print(f'downloading {f.name}')
    files.download(str(f))